# 01 — From attendance truth to noisy evidence

    A row is one simulated attempt, not an employee or image. We first decide
    whether the represented identity is correct, whether the presentation is live
    and whether the true location is inside 150 m. We then generate imperfect
    observable measurements. This makes truth independent of the decision rule.

    Run top to bottom. The example is deterministic and does not train a model.
    The published experiments use larger samples; all parameters remain explicit
    in `src/attendpro_paper2/simulation.py`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# The same notebook works from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/attendpro_paper2").exists())
thread_limit = threadpool_limits(limits=1)
pd.set_option("display.max_columns", 12)

In [ ]:
from attendpro_paper2 import FEATURES, REGIMES, evaluate, rule, simulate_attendpro, usable

seed = 20265001  # A teaching seed, separate from the paper's retained tests.
attempts = simulate_attendpro(5000, "nominal", seed)
display(attempts.head())
display(attempts.groupby("scenario").agg(
    n=("truth_valid_presence", "size"),
    valid_fraction=("truth_valid_presence", "mean"),
    average_true_distance=("true_distance_m", "mean")))

## Truth is not a predictor

    Genuine live outsiders are invalid even if their facial match is convincing.
    Presentation material is invalid even when it represents the enrolled identity.
    The model receives only the six observed columns below. It must not receive
    true distance, scenario names or the target itself.

In [ ]:
print("Observed model inputs:", FEATURES)
valid_scenarios = ["genuine_inside", "genuine_boundary_inside", "genuine_degraded_inside"]
expected_truth = attempts.scenario.isin(valid_scenarios) & (attempts.true_distance_m <= 150)
assert np.array_equal(attempts.truth_valid_presence.astype(bool), expected_truth)
display(attempts[FEATURES].describe().round(3))

## GPS accuracy is an uncertainty radius

    Larger `gps_accuracy_m` means a less precise reading. The reference geofence
    requires observed distance plus that radius to stay within 150 m. The plot
    shows errors in observed distance; it is not a simulation of forged coordinates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(attempts.true_distance_m, attempts.observed_distance_m,
                s=5, alpha=.2)
axes[0].plot([0, 450], [0, 450], color="black", linewidth=1)
axes[0].axvline(150, color="tab:red", linestyle="--")
axes[0].set(xlabel="True distance (m)", ylabel="Observed distance (m)",
            title="True and observed location evidence")
axes[1].hist(attempts.gps_accuracy_m, bins=35)
axes[1].set(xlabel="Reported uncertainty radius (m)", ylabel="Attempts",
            title="Nominal GPS accuracy distribution")
fig.tight_layout()
plt.show()

## Quality gates come before fusion

    Both GPS and capture gates must pass. Rejected legitimate attempts contribute
    to FRR even if a classifier could otherwise assign a high score. The reference
    conjunction then checks location, facial similarity, liveness and consistency.

In [ ]:
gate = usable(attempts)
valid = attempts.truth_valid_presence.astype(bool).to_numpy()
print("Legitimate attempts rejected by shared gates (%):", round(100 * (~gate[valid]).mean(), 2))
accepted = rule(attempts)
assert np.array_equal(accepted, attempts.M4_current_attendpro.astype(bool))
display(pd.DataFrame([evaluate(valid, accepted)]))
display(pd.DataFrame({"regime": list(REGIMES),
                      "GPS median (m)": [r["gps_median"] for r in REGIMES.values()],
                      "quality shift": [r["quality_shift"] for r in REGIMES.values()]}))

## Save attempts for a teammate

These are new teaching data. To generate the original nominal fit-0/batch-0
sample, call simulate_attendpro(20000, "nominal", seed=20262001).
No large attempt dataset is bundled in the light package.

In [ ]:
destination = ROOT / "runs/notebook01"
destination.mkdir(parents=True, exist_ok=True)
attempts.to_csv(destination / "teaching_attempts.csv", index=False)
print("Saved", len(attempts), "synthetic attempts.")

### Try next

    Generate `gps_noisy` or `face_degraded` with a new seed. Compare shared gate
    rejection and subgroup errors. Do not interpret these assumed distributions
    as fitted production data. Continue to Notebook 02 for learning and testing.